# Validación del modelo del bot XRP/USDT

Notebook de la **Fase 4 (validación)** del Proyecto Final Integrador.

Reutiliza las funciones de `validacion_rf.py` (mismo cálculo de variables que `bot_xrp_futuros.py`) para:

1. Descargar velas reales de 1 min de XRP_USDT (futuros, API pública de Gate.io) y armar velas de 2 min.
2. Entrenar el **Random Forest** del bot y compararlo con **KNN** y con modelos base.
3. Calcular **RMSE, MSE, MAE, MAPE, SMAPE, Variance Score** (regresión) y **matriz de confusión, Accuracy, Precision, Recall, F1** (clasificación Buy/Sell).
4. Aplicar el **test de Diebold-Mariano**.
5. Simular hiperparámetros del Random Forest.

> Requiere internet. Ejecutar desde la raíz del repositorio: `jupyter notebook notebooks/validacion_modelo.ipynb`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import validacion_rf as v
from IPython.display import Image, Markdown, display

## 1. Datos y variables
Partición **temporal** 80/20 sin mezclar, con purga de 3 velas entre train y test para evitar fuga de información.

In [ ]:
velas = v.velas_2m(v.descargar_velas_1m())
df = v.variables(velas)
train, test = v.dividir(df)
print(f"{len(df)} velas de 2 min | train {len(train)} | test {len(test)}")
df[v.FEATURES + ['close', 'precio_futuro', 'sube']].tail()

## 2. Correr la validación completa
`main()` entrena todos los modelos, guarda CSV y gráficas en `resultados/`.

In [ ]:
v.main()

## 3. Clasificación Buy/Sell — matriz de confusión y F1

In [ ]:
display(pd.read_csv(v.SALIDA / 'metricas_clasificacion.csv', index_col=0))
display(Image(v.SALIDA / 'matrices_confusion.png'))

## 4. Regresión — RMSE, MSE, MAE, MAPE, SMAPE, Variance Score

In [ ]:
display(pd.read_csv(v.SALIDA / 'metricas_regresion.csv', index_col=0))
display(Image(v.SALIDA / 'pronostico_vs_real.png'))

## 5. Test de Diebold-Mariano
$d_t = e_{1,t}^2 - e_{2,t}^2$, $DM = \bar d / \sqrt{\hat V(\bar d)}$ con varianza HAC de $h-1$ rezagos y corrección de Harvey-Leybourne-Newbold.
H0: ambos modelos tienen la misma precisión. DM < 0 → el modelo 1 tiene menor error.

In [ ]:
pd.read_csv(v.SALIDA / 'diebold_mariano.csv')

## 6. Simulación de hiperparámetros del Random Forest

In [ ]:
h = pd.read_csv(v.SALIDA / 'simulacion_hiperparametros.csv')
display(h.sort_values('RMSE').head(8))
display(Image(v.SALIDA / 'hiperparametros.png'))

## 7. Conclusiones
Ver la sección **Resultados de validación** del `README.md` y `resultados/resumen_validacion.md`.